# 0. Overview

This notebook analyzes ENSO-conditioned composite anomalies in **Mean Sea Level Pressure (MSLP)** over the **Indo-Pacific** and **Maritime Continent** domains.

Scientific objective:
- Separate mean-state change from El Niño / La Niña event responses in DJF MSLP anomalies.
- Use the same DJF definition, Niño3.4 thresholding, and period split as the source wind notebook.

# 1. Import Library

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import dask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'

# 2. Open Data & Pre-Process

#### Open ERA5 MSLP

In [ ]:
data_path = '../../external/ClimateData/era5-monthly/mslp_global_1979-2025.nc'
field_candidates = ['msl', 'mslp', 'mean sea level pressure', 'air_pressure_at_mean_sea_level']
output_name = 'mslp'
convert_mode = 'hpa'
pressure_level = None


def detect_data_var(ds, candidates):
    lowered = {name.lower(): name for name in ds.data_vars}
    for candidate in candidates:
        candidate = candidate.lower()
        for lowered_name, original_name in lowered.items():
            if candidate in lowered_name:
                return original_name
    if len(ds.data_vars) == 1:
        return next(iter(ds.data_vars))
    raise KeyError(
        f"Could not find a data variable matching {candidates!r}. Available variables: {list(ds.data_vars)!r}"
    )


def load_processed_field(path, candidates, output_name, convert_mode='none', pressure_level=None):
    ds = xr.open_dataset(path, chunks={'valid_time': 12})

    if 'pressure_level' in ds.coords or 'pressure_level' in ds.dims:
        if pressure_level is not None and pressure_level in np.asarray(ds['pressure_level'].values):
            ds = ds.sel(pressure_level=pressure_level)
        elif ds['pressure_level'].size == 1:
            ds = ds.isel(pressure_level=0)

    var_name = detect_data_var(ds, candidates)

    rename_map = {}
    if 'valid_time' in ds.coords or 'valid_time' in ds.dims:
        rename_map['valid_time'] = 'time'
    if 'latitude' in ds.coords or 'latitude' in ds.dims:
        rename_map['latitude'] = 'lat'
    if 'longitude' in ds.coords or 'longitude' in ds.dims:
        rename_map['longitude'] = 'lon'
    if rename_map:
        ds = ds.rename(rename_map)

    ds = ds.assign_coords(lon=(ds.lon % 360)).sortby('lon')
    ds = ds.sel(
        time=slice('1980-12-01', '2025-02-01'),
        lat=slice(31, -31),
        lon=slice(60, 291),
    )
    month_mask = ds.time.dt.month.isin([12, 1, 2])
    djf_year = xr.where(ds.time.dt.month == 12, ds.time.dt.year + 1, ds.time.dt.year)
    ds = ds.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))

    field = ds[var_name]
    units = str(field.attrs.get('units', '')).strip().lower()

    if convert_mode == 'hpa':
        if units in {'pa', 'pascal', 'pascals'}:
            field = field / 100.0
        field = field.assign_attrs(
            units='hPa',
            long_name='Mean sea level pressure',
            standard_name='air_pressure_at_mean_sea_level',
        )
    elif convert_mode == 'height_m':
        if units in {'m**2 s**-2', 'm^2 s^-2', 'm2 s^-2', 'm**2/s**-2'} or str(field.attrs.get('standard_name', '')).lower() == 'geopotential':
            field = field / 9.80665
        field = field.assign_attrs(
            units='m',
            long_name='Geopotential height at 850 hPa',
            standard_name='geopotential_height',
        )
    else:
        field = field.assign_attrs(units=field.attrs.get('units', ''))

    field = field.rename(output_name)
    print('Loaded variable:', repr(var_name))
    print('Converted field:', repr(output_name))
    print('Units:', field.attrs.get('units', 'unknown'))
    print('Time coverage:', str(field.time.values[0])[:10], 'to', str(field.time.values[-1])[:10])
    return field


field = load_processed_field(
    data_path,
    field_candidates,
    output_name,
    convert_mode=convert_mode,
    pressure_level=pressure_level,
)

#### Open NINO34 index

In [ ]:
nino34_path = '../../external/ClimateData/index-monthly/nino34.anom.csv'
nino34_column = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

df_nino34 = pd.read_csv(
    nino34_path,
    usecols=['Date', nino34_column],
    parse_dates=['Date'],
)
df_nino34 = df_nino34.set_index('Date').loc['1980-12-01':'2025-03-01']
df_nino34 = df_nino34[df_nino34.index.month.isin([12, 1, 2])].copy()
df_nino34['djf_year'] = df_nino34.index.year + (df_nino34.index.month == 12).astype('int8')
df_nino34['DJF'] = 'DJF' + df_nino34['djf_year'].astype(str)

#### Define Period

In [ ]:
full_years = np.arange(1981, 2026)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2026)

field_clim = field.sel(time=field.djf_year.isin(full_years))
field_past = field_clim.sel(time=field_clim.djf_year.isin(past_years))
field_recent = field_clim.sel(time=field_clim.djf_year.isin(recent_years))

nino34_clim = df_nino34[df_nino34['djf_year'].isin(full_years)].copy()
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()

period_coord = xr.DataArray(
    np.where(field_clim.djf_year <= 2006, 'past', 'recent'),
    coords={'time': field_clim.time},
    dims='time',
    name='period',
)
field_period = field_clim.assign_coords(period=period_coord)

field_full_plot, period_means = dask.compute(
    field_clim.mean('time'),
    field_period.groupby('period').mean('time'),
)

field_full_plot = field_full_plot.transpose('lat', 'lon')
field_past_plot = period_means.sel(period='past').transpose('lat', 'lon')
field_recent_plot = period_means.sel(period='recent').transpose('lat', 'lon')
field_diff_plot = field_recent_plot - field_past_plot


def plot_domain_panels(plots, extent, xticks, yticks, figsize):
    for data, title, cmap, cbar_label, levels, ticks, extend in plots:
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

        plot_data = data.reset_coords(drop=True)
        img = plot_data.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            levels=levels,
            extend=extend,
            add_colorbar=False,
            infer_intervals=True,
        )

        ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
        ax.set_xticks(xticks, crs=ccrs.PlateCarree())
        ax.set_yticks(yticks, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(direction='out', top=True, right=True, labelsize=14)
        ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
        ax.set_xlabel('')
        ax.set_ylabel('')

        cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
        cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
        cbar.set_label(cbar_label, fontsize=14, labelpad=10)
        cbar.ax.tick_params(labelsize=14)

        plt.show()

# 3. Define Area

#### Indo-Pacific domain

In [ ]:
# Define the extent and ticks
# ip = Indonesia-Pacific region
ip_extent = [60, 270, -30, 30]
ip_yticks = np.arange(-30, 31, 10)
ip_xticks = np.arange(60, 271, 30)

#### Maritime Continent domain

In [ ]:
# Define the extent and ticks
# mc: maritime continent
mc_extent = [80, 160, -20, 20]  # [lon_min, lon_max, lat_min, lat_max]
mc_yticks = np.arange(-20, 21, 10)
mc_xticks = np.arange(90, 161, 20)

# 4. Plot Composite

### A. Analisis El Nino

In [ ]:
# define el nino years based on DJF-mean nino34 index threshold
elnino_threshold = 0.5
nino34_clim_djf = nino34_clim.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_past_djf = nino34_past.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_recent_djf = nino34_recent.groupby('djf_year')[nino34_column].mean().reset_index()

nino34_clim_elnino = nino34_clim_djf[nino34_clim_djf[nino34_column] >= elnino_threshold]
nino34_past_elnino = nino34_past_djf[nino34_past_djf[nino34_column] >= elnino_threshold]
nino34_recent_elnino = nino34_recent_djf[nino34_recent_djf[nino34_column] >= elnino_threshold]

elnino_years_clim = sorted(nino34_clim_elnino['djf_year'].tolist())
elnino_years_past = sorted(nino34_past_elnino['djf_year'].tolist())
elnino_years_recent = sorted(nino34_recent_elnino['djf_year'].tolist())
print('El Nino DJF years (1981-2025):', elnino_years_clim)
print('El Nino DJF years (1981-2006):', elnino_years_past)
print('El Nino DJF years (2007-2025):', elnino_years_recent)

field_clim_elnino = field_clim.sel(time=field_clim.djf_year.isin(elnino_years_clim)).mean('time').transpose('lat', 'lon')
field_past_elnino = field_past.sel(time=field_past.djf_year.isin(elnino_years_past)).mean('time').transpose('lat', 'lon')
field_recent_elnino = field_recent.sel(time=field_recent.djf_year.isin(elnino_years_recent)).mean('time').transpose('lat', 'lon')

field_clim_elnino_anom = (field_clim_elnino - field_full_plot).reset_coords(drop=True)
field_past_elnino_anom = (field_past_elnino - field_past_plot).reset_coords(drop=True)
field_recent_elnino_anom = (field_recent_elnino - field_recent_plot).reset_coords(drop=True)
field_diff_elnino_anom = (field_recent_elnino_anom - field_past_elnino_anom).reset_coords(drop=True)

#### plot komposit indo pasifik

In [ ]:
plots = [
    (field_clim_elnino_anom, 'Komposit Anomali MSLP DJF 1981-2025 El Nino', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_past_elnino_anom, 'Komposit Anomali MSLP DJF 1981-2006 El Nino', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_recent_elnino_anom, 'Komposit Anomali MSLP DJF 2007-2025 El Nino', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_diff_elnino_anom, 'Selisih Komposit Anomali MSLP DJF P2 - P1 El Nino', 'RdYlBu', 'Selisih anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
]

plot_domain_panels(plots, ip_extent, ip_xticks, ip_yticks, figsize=(12, 6))

#### plot komposit maritime continent

In [ ]:
plots = [
    (field_clim_elnino_anom, 'Komposit Anomali MSLP DJF 1981-2025 El Nino', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_past_elnino_anom, 'Komposit Anomali MSLP DJF 1981-2006 El Nino', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_recent_elnino_anom, 'Komposit Anomali MSLP DJF 2007-2025 El Nino', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_diff_elnino_anom, 'Selisih Komposit Anomali MSLP DJF P2 - P1 El Nino', 'RdYlBu', 'Selisih anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
]

plot_domain_panels(plots, mc_extent, mc_xticks, mc_yticks, figsize=(10, 6))

### B. Analisis La Nina

In [ ]:
# define la nina years based on DJF-mean nino34 index threshold
lanina_threshold = -0.5

nino34_clim_lanina = nino34_clim_djf[nino34_clim_djf[nino34_column] <= lanina_threshold]
nino34_past_lanina = nino34_past_djf[nino34_past_djf[nino34_column] <= lanina_threshold]
nino34_recent_lanina = nino34_recent_djf[nino34_recent_djf[nino34_column] <= lanina_threshold]

lanina_years_clim = sorted(nino34_clim_lanina['djf_year'].tolist())
lanina_years_past = sorted(nino34_past_lanina['djf_year'].tolist())
lanina_years_recent = sorted(nino34_recent_lanina['djf_year'].tolist())
print('La Nina DJF years (1981-2025):', lanina_years_clim)
print('La Nina DJF years (1981-2006):', lanina_years_past)
print('La Nina DJF years (2007-2025):', lanina_years_recent)

field_clim_lanina = field_clim.sel(time=field_clim.djf_year.isin(lanina_years_clim)).mean('time').transpose('lat', 'lon')
field_past_lanina = field_past.sel(time=field_past.djf_year.isin(lanina_years_past)).mean('time').transpose('lat', 'lon')
field_recent_lanina = field_recent.sel(time=field_recent.djf_year.isin(lanina_years_recent)).mean('time').transpose('lat', 'lon')

field_clim_lanina_anom = (field_clim_lanina - field_full_plot).reset_coords(drop=True)
field_past_lanina_anom = (field_past_lanina - field_past_plot).reset_coords(drop=True)
field_recent_lanina_anom = (field_recent_lanina - field_recent_plot).reset_coords(drop=True)
field_diff_lanina_anom = (field_recent_lanina_anom - field_past_lanina_anom).reset_coords(drop=True)

#### plot komposit indo pasifik

In [ ]:
plots = [
    (field_clim_lanina_anom, 'Komposit Anomali MSLP DJF 1981-2025 La Nina', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_past_lanina_anom, 'Komposit Anomali MSLP DJF 1981-2006 La Nina', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_recent_lanina_anom, 'Komposit Anomali MSLP DJF 2007-2025 La Nina', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_diff_lanina_anom, 'Selisih Komposit Anomali MSLP DJF P2 - P1 La Nina', 'RdYlBu', 'Selisih anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
]

plot_domain_panels(plots, ip_extent, ip_xticks, ip_yticks, figsize=(12, 6))

#### plot komposit maritime continent

In [ ]:
plots = [
    (field_clim_lanina_anom, 'Komposit Anomali MSLP DJF 1981-2025 La Nina', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_past_lanina_anom, 'Komposit Anomali MSLP DJF 1981-2006 La Nina', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_recent_lanina_anom, 'Komposit Anomali MSLP DJF 2007-2025 La Nina', 'RdYlBu', 'Anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
    (field_diff_lanina_anom, 'Selisih Komposit Anomali MSLP DJF P2 - P1 La Nina', 'RdYlBu', 'Selisih anomali MSLP (hPa)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
]

plot_domain_panels(plots, mc_extent, mc_xticks, mc_yticks, figsize=(10, 6))